In [1]:
"""
CTC Loss (Connectionist Temporal Classification)
===================================================

THIS is the loss function used in most OCR/handwriting recognition 
systems - directly relevant to your Devanagari work later.

Problem CTC solves:
  In OCR, you have an image of a WORD (variable length text),
  but you don't know which pixel-columns correspond to which character.
  
  Standard CrossEntropy needs: exact alignment between each prediction 
  and each true label (position 1 = label 1, position 2 = label 2, etc.)
  
  But in a manuscript image, the model doesn't know in advance:
  - How many pixel-columns per character
  - Where one character ends and another begins
  - Manuscripts have variable spacing, distortion, connected characters
"""
import torch
import torch.nn as nn

print("=" * 70)
print("WHY CTC LOSS EXISTS")
print("=" * 70)

WHY CTC LOSS EXISTS


In [2]:
print("""
Example: Image of handwritten word "cat"

Model processes image left-to-right, outputs a prediction at EVERY 
timestep (imagine 8 timesteps, sliding a window across the image):

Timestep:  1    2    3    4    5    6    7    8
Predicted: c    c    c    a    a    -    t    t

Notice:
  - 'c' takes 3 timesteps (character is wide, or slowly scanned)
  - 'a' takes 2 timesteps  
  - '-' means blank (no character, e.g. space between letters)
  - 't' takes 2 timesteps

CTC has a special "blank" token and COLLAPSE RULE:
  1. Merge consecutive repeated characters: "cccaa-tt" → "ca-t"
  2. Remove blank tokens: "ca-t" → "cat"

Result: "cat" ✓ — without ever needing to know EXACTLY which 
timestep boundary belongs to which character!

This is why CTC is used for OCR/speech recognition — it lets the 
model output a prediction at every timestep without needing 
pre-aligned training labels.
""")



Example: Image of handwritten word "cat"

Model processes image left-to-right, outputs a prediction at EVERY 
timestep (imagine 8 timesteps, sliding a window across the image):

Timestep:  1    2    3    4    5    6    7    8
Predicted: c    c    c    a    a    -    t    t

Notice:
  - 'c' takes 3 timesteps (character is wide, or slowly scanned)
  - 'a' takes 2 timesteps  
  - '-' means blank (no character, e.g. space between letters)
  - 't' takes 2 timesteps

CTC has a special "blank" token and COLLAPSE RULE:
  1. Merge consecutive repeated characters: "cccaa-tt" → "ca-t"
  2. Remove blank tokens: "ca-t" → "cat"

Result: "cat" ✓ — without ever needing to know EXACTLY which 
timestep boundary belongs to which character!

This is why CTC is used for OCR/speech recognition — it lets the 
model output a prediction at every timestep without needing 
pre-aligned training labels.



In [3]:
print("=" * 70)
print("MINIMAL WORKING EXAMPLE")
print("=" * 70)

# CTC Loss usage pattern (structure only — real OCR needs real sequence data)
ctc_loss = nn.CTCLoss(blank=0)  # index 0 reserved for blank token

# Simulated model output: (timesteps, batch, num_classes)
# 5 timesteps, batch of 1, 4 possible classes (0=blank, 1='c', 2='a', 3='t')
T, N, C = 5, 1, 4
log_probs = torch.randn(T, N, C).log_softmax(2)  # model's raw predictions

# True label: "cat" = [1, 2, 3] (no blanks needed in the TARGET, only in prediction)
targets = torch.tensor([1, 2, 3])

input_lengths = torch.tensor([T])       # how many timesteps the model produced
target_lengths = torch.tensor([3])      # true label length ("cat" = 3 chars)

loss = ctc_loss(log_probs, targets, input_lengths, target_lengths)
print(f"\nCTC Loss for this example: {loss.item():.4f}")

MINIMAL WORKING EXAMPLE

CTC Loss for this example: 2.0709


In [4]:
print("""
Key inputs CTC needs:
  log_probs:      model's per-timestep class probabilities (log space)
  targets:        the TRUE label sequence, no blanks, no repeats needed
  input_lengths:  how many timesteps the model actually produced
  target_lengths: how many characters the true label has

WHERE THIS MATTERS FOR YOUR DEVANAGARI WORK:
  Your line-extraction + character-recognition pipeline is EXACTLY 
  this problem. Instead of segmenting each character perfectly first,
  CTC lets you train end-to-end: image → sequence of characters,
  without needing pixel-perfect character boundaries during training.
  
  This is likely more robust for manuscripts with distortion, 
  connected characters (common in Devanagari due to the horizontal 
  headline connecting characters), and inconsistent spacing.
""")


Key inputs CTC needs:
  log_probs:      model's per-timestep class probabilities (log space)
  targets:        the TRUE label sequence, no blanks, no repeats needed
  input_lengths:  how many timesteps the model actually produced
  target_lengths: how many characters the true label has

WHERE THIS MATTERS FOR YOUR DEVANAGARI WORK:
  Your line-extraction + character-recognition pipeline is EXACTLY 
  this problem. Instead of segmenting each character perfectly first,
  CTC lets you train end-to-end: image → sequence of characters,
  without needing pixel-perfect character boundaries during training.
  
  This is likely more robust for manuscripts with distortion, 
  connected characters (common in Devanagari due to the horizontal 
  headline connecting characters), and inconsistent spacing.

